In [1]:
!pip install transformers accelerate torch pandas tqdm

In [2]:
import re
import pandas as pd
import torch
from tqdm import tqdm
from transformers import pipeline
from google.colab import files

In [3]:
import os

REPO_URL = "https://github.com/simona-wang/negotiation_arena.git"
PROJECT_DIR = "/content/negotiation_arena"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULTS_DIR)

Cloning into '/content/negotiation_arena'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 210 (delta 136), reused 193 (delta 119), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 1.07 MiB | 17.88 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Project directory: /content/negotiation_arena
Results directory: /content/negotiation_arena/results


In [4]:
human_data_path = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "human_negotiations.csv"
)

human_df = pd.read_csv(human_data_path)

print(human_df.shape)
human_df.head()

(4993, 8)


,dialogue_id,split,turn_id,role,text,act,issue,value
0,0,train,0,Candidate,Hello. I would like to discuss the issues of m...,Greet,NaN,True
1,0,train,1,Candidate,I would like a position of project manager,Offer,Job Description,Project Manager
2,0,train,2,Employer,We do not have any project manager positions o...,Reject,Job Description,Project Manager
3,0,train,3,Candidate,I want a position of project manager,Offer,Job Description,Project Manager
4,0,train,4,Employer,I do have programmer positions open with a str...,Offer,Job Description,Programmer


In [5]:
human_df.shape

(4993, 8)

In [6]:
human_df["dialogue_id"].nunique()

105

In [7]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [8]:
def generate_response(system_prompt, user_message, max_new_tokens=180):
    prompt = f"""
<|system|>
{system_prompt}

<|user|>
{user_message}

<|assistant|>
"""

    output = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_full_text=False,
        pad_token_id=pipe.tokenizer.eos_token_id
    )

    return output[0]["generated_text"].strip()

In [9]:
NEGOTIATION_ACTS = ["Offer", "Reject", "Query", "Accept"]

def get_first_negotiation_act(human_df, dialogue_id):
    dialogue = human_df[
        (human_df["dialogue_id"].astype(str) == str(dialogue_id)) &
        (human_df["act"].isin(NEGOTIATION_ACTS))
    ].sort_values("turn_id")

    if dialogue.empty:
        return None

    return dialogue.iloc[0]

In [10]:
#test for first act
first_act = get_first_negotiation_act(human_df, human_df["dialogue_id"].iloc[0])
first_act

,1
dialogue_id,0
split,train
turn_id,1
role,Candidate
text,I would like a position of project manager
act,Offer
issue,Job Description
value,Project Manager


In [11]:
def build_agent_prompt(role):
    return f"""
You are the {role} in a job contract negotiation.

You are continuing a negotiation that started from a real human negotiation act.

Your goal:
- negotiate realistically
- respond according to your role
- use offers, counteroffers, rejections, questions, or acceptance when appropriate
- avoid repeating the same message
- try to reach an agreement if possible
- if no progress is made after repeated disagreement, end the negotiation

Use one of these dialogue acts:
Offer, Accept, Reject, Query, Quit, Other

Rules:
- If you accept the other party's proposal, write ACT: Accept and DECISION: accept.
- If the negotiation is clearly stuck or no compromise is possible, write ACT: Quit and DECISION: quit.
- If you are still negotiating, write DECISION: continue.

Reply ONLY in this format:

MESSAGE: your short negotiation message
ACT: Offer / Accept / Reject / Query / Quit / Other
ISSUE: negotiated issue or NONE
VALUE: proposed value or NONE
DECISION: continue / accept / quit
"""

In [12]:
def parse_llm_act_response(text):
    result = {
        "message": None,
        "act": None,
        "issue": None,
        "value": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("ACT:"):
            result["act"] = line.replace("ACT:", "").strip()

        elif line.startswith("ISSUE:"):
            value = line.replace("ISSUE:", "").strip()
            result["issue"] = None if value.upper() == "NONE" else value

        elif line.startswith("VALUE:"):
            value = line.replace("VALUE:", "").strip()
            result["value"] = None if value.upper() == "NONE" else value

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [13]:
def build_seed_message(first_act):
    return f"""
The following is the first substantive negotiation act from a human job negotiation.

Speaker: {first_act["role"]}
Utterance: {first_act["text"]}
Dialogue act: {first_act["act"]}
Issue: {first_act["issue"]}
Value: {first_act["value"]}

Continue the negotiation from this point.
"""

In [14]:
def get_next_speaker(role):
    if role == "Candidate":
        return "Employer"
    elif role == "Employer":
        return "Candidate"
    else:
        return "Employer"

In [15]:
def run_human_seeded_simulation(human_df, dialogue_id, max_cap=20):
    first_act = get_first_negotiation_act(human_df, dialogue_id)

    if first_act is None:
        return [], {
            "dialogue_id": dialogue_id,
            "outcome": "NoSeed",
            "n_turns": 0,
            "human_original_turns": 0,
            "seed_role": None,
            "seed_act": None,
            "seed_issue": None,
            "seed_value": None
        }

    human_dialogue = human_df[
        human_df["dialogue_id"].astype(str) == str(dialogue_id)
    ]

    human_length = human_dialogue["turn_id"].nunique()
    max_turns = min(human_length, max_cap)

    seed_message = build_seed_message(first_act)

    conversation_log = []

    conversation_log.append({
        "dialogue_id": dialogue_id,
        "turn": 0,
        "speaker": first_act["role"],
        "text": first_act["text"],
        "act": first_act["act"],
        "issue": first_act["issue"],
        "value": first_act["value"],
        "decision": "seed",
        "is_seed": True
    })

    current_message = seed_message
    current_speaker = get_next_speaker(first_act["role"])

    outcome = None

    for turn in range(1, max_turns + 1):

        system_prompt = build_agent_prompt(current_speaker)

        llm_text = generate_response(
            system_prompt,
            current_message
        )

        parsed = parse_llm_act_response(llm_text)

        conversation_log.append({
            "dialogue_id": dialogue_id,
            "turn": turn,
            "speaker": current_speaker,
            "text": llm_text,
            "act": parsed["act"],
            "issue": parsed["issue"],
            "value": parsed["value"],
            "decision": parsed["decision"],
            "is_seed": False
        })

        if parsed["decision"] == "accept" or parsed["act"] == "Accept":
            outcome = "Agreement"
            break

        if parsed["decision"] == "quit" or parsed["act"] == "Quit":
            outcome = "Failure"
            break

        recent_context = conversation_log[-6:]

        context_text = ""
        for row in recent_context:
            context_text += f'{row["speaker"]}: {row["text"]}\n'

        next_speaker = get_next_speaker(current_speaker)

        current_message = f"""
Continue the following job negotiation.

Recent conversation:
{context_text}

Now reply as {next_speaker}.
"""

        current_speaker = next_speaker

    if outcome is None:
        recent_non_seed = [
            row for row in conversation_log[-6:]
            if row["is_seed"] == False
        ]

        recent_acts = [
            str(row["act"]).lower()
            for row in recent_non_seed
            if row["act"] is not None
        ]

        recent_decisions = [
            str(row["decision"]).lower()
            for row in recent_non_seed
            if row["decision"] is not None
        ]

        recent_texts = [
            str(row["text"]).lower()
            for row in recent_non_seed
            if row["text"] is not None
        ]

        recent_text = " ".join(recent_texts)

        impasse_keywords = [
            "cannot accept",
            "can't accept",
            "not acceptable",
            "too low",
            "too high",
            "below my minimum",
            "above our maximum",
            "no agreement",
            "unable to agree",
            "end the negotiation",
            "walk away"
        ]

        if (
            "reject" in recent_acts
            or recent_decisions.count("continue") >= 4
            or any(keyword in recent_text for keyword in impasse_keywords)
        ):
            outcome = "Impasse"
        else:
            outcome = "Timeout"

    outcome_row = {
        "dialogue_id": dialogue_id,
        "outcome": outcome,
        "n_turns": len(conversation_log),
        "human_original_turns": human_length,
        "seed_role": first_act["role"],
        "seed_act": first_act["act"],
        "seed_issue": first_act["issue"],
        "seed_value": first_act["value"]
    }

    return conversation_log, outcome_row

In [16]:
##test on 3 dialogues

test_dialogues = human_df["dialogue_id"].astype(str).unique()[:3]

test_turns = []
test_outcomes = []

for dialogue_id in test_dialogues:
    print("Running test dialogue:", dialogue_id)

    log, outcome = run_human_seeded_simulation(
        human_df,
        dialogue_id,
        max_cap=10
    )

    test_turns.extend(log)
    test_outcomes.append(outcome)

test_turns_df = pd.DataFrame(test_turns)
test_outcomes_df = pd.DataFrame(test_outcomes)

test_outcomes_df

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'temperature', 'max_new_tokens', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running test dialogue: 0


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running test dialogue: 1


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Running test dialogue: 10


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

,dialogue_id,outcome,n_turns,human_original_turns,seed_role,seed_act,seed_issue,seed_value
0,0,Agreement,4,55,Candidate,Offer,Job Description,Project Manager
1,1,Timeout,11,27,Candidate,Offer,Job Description,Project Manager
2,10,Timeout,11,63,Candidate,Offer,Job Description,Project Manager


In [17]:
test_turns_df.head(20)

,dialogue_id,turn,speaker,text,act,issue,value,decision,is_seed
0,0,0,Candidate,I would like a position of project manager,Offer,Job Description,Project Manager,seed,True
1,0,1,Employer,Speaker: Candidate\nUtterance: Based on the pr...,None,None,None,None,False
2,0,2,Candidate,MESSAGE: Thank you for your interest in the pr...,Offer / Accept / Reject / Query / Quit / Other,Job Description / Project Manager / No Issue,Project Manager / No Value,continue / accept / reject / query / quit,False
3,0,3,Employer,MESSAGE: Thank you for your interest in the pr...,Offer / Accept / Reject / Query / Quit / Other,Job Description / Project Manager / No Issue,Project Manager / No Value,accept,False
4,1,0,Candidate,I am expecting a position of project manager,Offer,Job Description,Project Manager,seed,True
5,1,1,Employer,Speaker: Candidate\nUtterance: I am expecting ...,None,None,None,None,False
6,1,2,Candidate,Speaker: Candidate\nUtterance: I appreciate yo...,None,None,None,None,False
7,1,3,Employer,Speaker: Employer\nUtterance: Candidate\n\nUtt...,None,None,None,None,False
8,1,4,Candidate,Candidate: Thank you for your interest in the ...,None,None,None,None,False
9,1,5,Employer,Candidate: Speaker: Candidate\nUtterance: I ap...,None,None,None,None,False


In [18]:
import random

random.seed(42)

all_dialogues = human_df["dialogue_id"].astype(str).unique().tolist()

sample_size = 20

sample_dialogues = random.sample(
    all_dialogues,
    min(sample_size, len(all_dialogues))
)

print("Number of sampled dialogues:", len(sample_dialogues))
print(sample_dialogues)

Number of sampled dialogues: 20
['78', '17', '100', '9', '36', '32', '3', '2', '16', '82', '67', '14', '72', '53', '101', '29', '30', '62', '74', '69']


In [19]:
human_seeded_turns = []
human_seeded_outcomes = []

for dialogue_id in tqdm(sample_dialogues):
    log, outcome = run_human_seeded_simulation(
        human_df,
        dialogue_id,
        max_cap=20
    )

    human_seeded_turns.extend(log)
    human_seeded_outcomes.append(outcome)

    pd.DataFrame(human_seeded_turns).to_csv(
       os.path.join(RESULTS_DIR, "human_seeded_llm_turns_sample.csv"),
        index=False
    )

    pd.DataFrame(human_seeded_outcomes).to_csv(
        os.path.join(RESULTS_DIR, "human_seeded_llm_outcomes_sample.csv"),
        index=False
    )

human_seeded_turns_df = pd.DataFrame(human_seeded_turns)
human_seeded_outcomes_df = pd.DataFrame(human_seeded_outcomes)

human_seeded_outcomes_df

  0%|          | 0/20 [00:00<?, ?it/s]Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  5%|▌         | 1/20 [00:11<03:34, 11.31s/it]Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more infor

,dialogue_id,outcome,n_turns,human_original_turns,seed_role,seed_act,seed_issue,seed_value
0,78,Agreement,3,47,Candidate,Offer,Job Description,Project Manager
1,17,Agreement,4,48,Candidate,Offer,Salary,"120,000 USD"
2,100,Agreement,10,43,Candidate,Offer,Job Description,Project Manager
3,9,Agreement,9,23,Candidate,Offer,Salary,"120,000 USD"
4,36,Impasse,19,18,Candidate,Offer,Job Description,Project Manager
5,32,Timeout,21,27,Candidate,Offer,Job Description,Project Manager
6,3,Agreement,3,32,Candidate,Offer,Salary,"120,000 USD"
7,2,Agreement,8,16,Candidate,Offer,Salary,"120,000 USD"
8,16,Impasse,15,14,Candidate,Offer,Job Description,Project Manager
9,82,Timeout,21,29,Candidate,Offer,Job Description,Project Manager


In [20]:
human_seeded_outcomes_df["outcome"].value_counts()

,count
outcome,
Agreement,10
Timeout,8
Impasse,2


In [21]:
human_seeded_outcomes_df["n_turns"].describe()

,n_turns
count,20.000000
mean,12.400000
std,8.394234
min,2.000000
25%,3.000000
50%,12.500000
75%,21.000000
max,21.000000


In [22]:
human_seeded_outcomes_df.head()

,dialogue_id,outcome,n_turns,human_original_turns,seed_role,seed_act,seed_issue,seed_value
0,78,Agreement,3,47,Candidate,Offer,Job Description,Project Manager
1,17,Agreement,4,48,Candidate,Offer,Salary,"120,000 USD"
2,100,Agreement,10,43,Candidate,Offer,Job Description,Project Manager
3,9,Agreement,9,23,Candidate,Offer,Salary,"120,000 USD"
4,36,Impasse,19,18,Candidate,Offer,Job Description,Project Manager


In [23]:
human_seeded_outcomes_df["turn_difference"] = (
    human_seeded_outcomes_df["n_turns"]
    - human_seeded_outcomes_df["human_original_turns"]
)

human_seeded_outcomes_df[[
    "dialogue_id",
    "outcome",
    "n_turns",
    "human_original_turns",
    "turn_difference"
]].head()

,dialogue_id,outcome,n_turns,human_original_turns,turn_difference
0,78,Agreement,3,47,-44
1,17,Agreement,4,48,-44
2,100,Agreement,10,43,-33
3,9,Agreement,9,23,-14
4,36,Impasse,19,18,1


In [24]:
human_seeded_outcomes_df[["n_turns", "human_original_turns"]].mean()

,0
n_turns,12.40
human_original_turns,29.65


In [25]:
from google.colab import files

files.download(os.path.join(RESULTS_DIR, "human_seeded_llm_turns_sample.csv"))
files.download(os.path.join(RESULTS_DIR, "human_seeded_llm_outcomes_sample.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>